In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor, Pool

cols = [
    # Для идентификации
    "date",

    # таргет
    "units",

    # прошлое
    "units_yesterday", "units_prev_week",
    # "rolling_mean_4w", - бесполезный.

    # категориальные
    "store_code", "store_item_code",

    # погода (float32)
    "tmax", "tmin", "tavg", "depart", "dewpoint", "wetbulb", "heat", "cool",
    "sunrise", "sunset",
    "snowfall", "preciptotal", "stnpressure", "sealevel",
    "resultspeed", "resultdir", "avgspeed",

    # календарь и флаги (int16)
    "year", "week", "BCFG", "BLDU", "BLSN", "BR", "DU", "DZ", "FG", "FU",
    "FZDZ", "FZFG", "FZRA", "GR", "GS", "HZ", "MIFG", "PL", "PRFG", "RA",
    "SG", "SN", "SQ", "TS", "TSRA", "TSSN", "UP", "VCFG", "VCTS",
    "day_of_week", "month", "is_weekend", "is_holiday",
    "rain_streak", "dry_streak",

    # look‑ahead
    "avg_temp_next_day", "rain_next_day", "days_to_holiday"
]

dtypes = {
    # целевой
    "units": "int16",        # -32 768 … 32 767

    # Для идентификации
    "date": "object",

    # прошлое → float32
    **{c: "float32" for c in [
        "units_yesterday", "units_prev_week", "rolling_mean_4w",
    ]},
    

    # категориальные коды
    "store_code": "category",
    "store_item_code": "category",

    # погода → float32
    **{c: "float32" for c in [
        "tmax","tmin","tavg","depart","dewpoint","wetbulb","heat","cool",
        "sunrise","sunset",
        "snowfall","preciptotal","stnpressure","sealevel",
        "resultspeed","resultdir","avgspeed",
        "avg_temp_next_day","rain_next_day",
    ]},

    # календарные/флаговые → int16
    **{c: "int16" for c in [
        "year","week","day_of_week","month",
        "is_weekend","is_holiday","rain_streak","dry_streak",
        "BCFG","BLDU","BLSN","BR","DU","DZ","FG","FU","FZDZ","FZFG",
        "FZRA","GR","GS","HZ","MIFG","PL","PRFG","RA","SG","SN","SQ",
        "TS","TSRA","TSSN","UP","VCFG","VCTS", "days_to_holiday",
    ]},
}

full_table_df = pd.read_csv(
    "./data/big_full_table.csv",
    usecols=cols,
    dtype=dtypes,
)
full_table_df['date'] = pd.to_datetime(full_table_df['date'])

In [8]:
full_table_df['date'].max()

Timestamp('2014-10-31 00:00:00')

### Обучение модели

In [3]:
# ---------- 1. подготовка X / y  ----------
# full_table_df уже в памяти (из предыдущего шага)
# dates = full_table_df["date"]
y = full_table_df["units"]
X = full_table_df.drop(columns=["units"])

cat_cols = ["store_code", "store_item_code"]  # CatBoost поймёт сам

# ---------- 2. выделение последних 2 недель для теста ----------
cutoff_date = pd.to_datetime("2014-10-17")  # 14 дней до 2014-10-31
# cutoff_date = pd.to_datetime("2014-10-01")  # 14 дней до 2014-10-31
is_test = full_table_df["date"] > cutoff_date

date_test = X[is_test]["date"]
X_test = X[is_test].drop(columns=["date"])
y_test = y[is_test]

X_remain = X[~is_test].drop(columns=["date"])
y_remain = y[~is_test]

# ---------- 3. train / valid ----------
X_train, X_valid, y_train, y_valid = train_test_split(
    X_remain, y_remain, test_size=0.2, random_state=42
)

In [4]:

# ---------- 3. CatBoost ----------
train_pool  = Pool(X_train, y_train, cat_features=cat_cols)
valid_pool  = Pool(X_valid, y_valid, cat_features=cat_cols)
test_pool   = Pool(X_test,  y_test,  cat_features=cat_cols)

model = CatBoostRegressor(
    loss_function="RMSE",         # будем минимизировать RMSE
    eval_metric="RMSE",           # лог‑RMSE посчитаем сами
    learning_rate=0.05,
    depth=8,
    iterations=3000,
    # iterations=1000,
    early_stopping_rounds=200,
    random_seed=42,
    verbose=200
)
model.fit(train_pool, eval_set=valid_pool)

# ---------- 4. оценка RMSLE ----------

y_pred = model.predict(test_pool).clip(min=0)
rmsle  = np.sqrt(mean_squared_log_error(y_test, y_pred))
print(f"RMSLE on test = {rmsle:.4f}")

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE on test = {rmse:.4f}")

0:	learn: 35.7470633	test: 46.8109710	best: 46.8109710 (0)	total: 272ms	remaining: 13m 35s
200:	learn: 13.3706794	test: 33.2916554	best: 33.2916554 (200)	total: 46.7s	remaining: 10m 50s
400:	learn: 12.4979355	test: 33.1433490	best: 33.1433490 (400)	total: 1m 31s	remaining: 9m 55s
600:	learn: 11.9131924	test: 33.0736642	best: 33.0736642 (600)	total: 2m 15s	remaining: 8m 58s
800:	learn: 11.4512955	test: 33.0239220	best: 33.0234452 (795)	total: 2m 58s	remaining: 8m 8s
1000:	learn: 11.0804896	test: 32.9882946	best: 32.9881953 (998)	total: 3m 41s	remaining: 7m 21s
1200:	learn: 10.7695691	test: 32.9635529	best: 32.9635529 (1200)	total: 4m 24s	remaining: 6m 35s
1400:	learn: 10.4951736	test: 32.9500270	best: 32.9495373 (1375)	total: 5m 6s	remaining: 5m 50s
1600:	learn: 10.2761541	test: 32.9427063	best: 32.9426321 (1597)	total: 5m 50s	remaining: 5m 5s
1800:	learn: 10.0777782	test: 32.9351249	best: 32.9350438 (1797)	total: 6m 40s	remaining: 4m 26s
2000:	learn: 9.8814787	test: 32.9295448	best: 32

In [5]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print(f"R² on test = {r2:.4f}")

R² on test = 0.8447


In [6]:
df = pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'date': date_test})
print(df.sample(40))

            y_pred  y_test       date
232873    0.179746       0 2014-10-18
233834   16.424387      17 2014-10-22
235618    0.000000       0 2014-10-30
235586   13.418508       9 2014-10-30
232834  135.346628     125 2014-10-18
234270    0.938601       1 2014-10-24
234348   57.472311     117 2014-10-25
235340    0.710713       0 2014-10-29
234469    0.000000       0 2014-10-25
233718    0.000000       0 2014-10-22
233523   28.478339      14 2014-10-21
232902    0.156241       0 2014-10-18
234332    0.058644       0 2014-10-25
232992    0.000000       0 2014-10-18
235717   95.451399     113 2014-10-30
232961   51.733945      39 2014-10-18
235067    2.252767       5 2014-10-28
233359  195.869755     241 2014-10-20
233510   52.448103      64 2014-10-21
235069    0.122439       0 2014-10-28
233076   70.898042      67 2014-10-19
234619    0.392049       0 2014-10-26
233396    3.060999       2 2014-10-20
234889   53.765073      15 2014-10-27
235214   35.804207      22 2014-10-28
234993    0.

In [21]:
import pandas as pd
import matplotlib.pyplot as plt

# ── 1. числовой вывод
importances = model.get_feature_importance(train_pool, type="PredictionValuesChange")
feat_names  = X_train.columns

imp_df = (
    pd.DataFrame({"feature": feat_names, "importance": importances})
      .sort_values("importance", ascending=False)
)

print(imp_df.head(20))   # топ‑20 в консоль


              feature  importance
2     units_yesterday   29.466343
3     units_prev_week   22.972018
1     store_item_code    5.524464
50        day_of_week    3.851140
16        stnpressure    3.415536
58    days_to_holiday    3.257297
17           sealevel    2.834928
19          resultdir    2.117419
21               year    2.088160
22               week    2.068444
0          store_code    2.043223
9             wetbulb    1.927825
51              month    1.840241
13             sunset    1.609161
18        resultspeed    1.431440
56  avg_temp_next_day    1.416673
4                tmax    1.279764
7              depart    1.247146
20           avgspeed    1.233116
55         dry_streak    1.059168


Мнение чата:

| Позиция          | RMSLE (≈)       | Что обычно отличает команды                                                 |
| ---------------- | --------------- | --------------------------------------------------------------------------- |
| 🥇 1‑е место     | **0.38 – 0.44** | сильная фич‑инженерия (лаги, сезонность, промо), тонкий hyper‑opt, ансамбли |
| 🥈 Топ‑3         | **0.44 – 0.48** | 1 х градиентный бустинг + набор умных признаков                             |
| 🥉 Топ‑10        | **0.48 – 0.55** | базовый бустинг, минимальный тюнинг, ограниченное число фич                 |
| Середина таблицы | 0.55 – 0.70     | «из коробки» модели, мало фич                                               |

У тебя уже 0.494 — граница входа в условный топ‑10. 
Оптимистичный прогноз: доведёшь до 0.42 – 0.45 — и шансы на победу вполне реальные.

Где ещё взять пару сотых:

- Лаги + rolling‑median по погоде (t‑7, t‑14 для tavg, precip).
- Target encoding категорий: item_nbr → средняя продажа по товару, store_nbr → сезонный коэффициент.
- Hyper‑opt (Optuna, 200 итераций) для depth, l2_leaf_reg, bagging_temperature.
- Blending CatBoost + LightGBM + Linear Reg (по лог‑таргету).

В сумме эти штрихи обычно дают −0.03 … −0.06 RMSLE.

Так что ориентир «< 0.45» держи как цель.

In [22]:
model.save_model("../ml-models/CatBoost v1.cbm")